In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [2]:
data = pd.read_csv("shop_smart_ecommerce.csv")
data.sample(5)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
5935,1,129.0,1,26.70,28,1392.100000,0.031111,0.056333,0.000000,0.0,Sep,3,2,7,13,Returning_Visitor,True,False
11682,6,213.5,0,0.00,11,303.500000,0.000000,0.016667,87.902961,0.0,Dec,8,13,9,20,Other,False,True
11399,4,54.5,0,0.00,74,1599.994589,0.000000,0.002632,67.266161,0.0,Dec,2,4,6,2,Returning_Visitor,False,True
8306,4,159.0,2,27.75,33,821.682540,0.000617,0.019343,0.000000,0.0,Dec,2,2,1,2,Returning_Visitor,False,True
6394,3,145.2,0,0.00,28,425.096905,0.000000,0.002381,0.000000,0.0,Oct,3,2,7,11,New_Visitor,True,False


In [3]:
data["VisitorType"].value_counts()

VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85
Name: count, dtype: int64

In [4]:
data["Product_Page_Duration"] = data["ProductRelated"]*data["ProductRelated_Duration"]

In [5]:
data

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,Product_Page_Duration
0,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Feb,1,1,1,1,Returning_Visitor,False,False,0.000000
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.000000,0.0,Feb,2,2,1,2,Returning_Visitor,False,False,128.000000
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Feb,4,1,9,3,Returning_Visitor,False,False,0.000000
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.000000,0.0,Feb,3,2,2,4,Returning_Visitor,False,False,5.333333
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.000000,0.0,Feb,3,3,1,4,Returning_Visitor,True,False,6275.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,0.007143,0.029031,12.241717,0.0,Dec,4,6,1,1,Returning_Visitor,True,False,94540.958351
12326,0,0.0,0,0.0,5,465.750000,0.000000,0.021333,0.000000,0.0,Nov,3,2,1,8,Returning_Visitor,True,False,2328.750000
12327,0,0.0,0,0.0,6,184.250000,0.083333,0.086667,0.000000,0.0,Nov,3,2,1,13,Returning_Visitor,True,False,1105.500000
12328,4,75.0,0,0.0,15,346.000000,0.000000,0.021053,0.000000,0.0,Nov,2,2,3,11,Returning_Visitor,False,False,5190.000000


In [6]:
data["Revenue"].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

In [7]:
x = data.drop("Revenue",axis = 1)
y = data["Revenue"].map({True:1,False:0})

In [8]:
print(y)

0        0
1        0
2        0
3        0
4        0
        ..
12325    0
12326    0
12327    0
12328    0
12329    0
Name: Revenue, Length: 12330, dtype: int64


In [9]:
num = x.select_dtypes(include=["int64","float64"]).columns
cat = x.select_dtypes(include=["object","category"]).columns

In [10]:
print(cat)
print(num)

Index(['Month', 'VisitorType'], dtype='object')
Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType',
       'Product_Page_Duration'],
      dtype='object')


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),num),
        ("cat",OneHotEncoder(),cat)
    ]
)

In [12]:
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

In [13]:
tree = DecisionTreeClassifier(random_state = 42)

path = tree.cost_complexity_pruning_path(preprocessor.fit_transform(x_train),y_train)
alphas = path.ccp_alphas
print(alphas)

[0.00000000e+00 5.72724113e-05 6.23869237e-05 6.50826550e-05
 6.55980154e-05 6.60140704e-05 6.67999522e-05 6.69166673e-05
 6.70578197e-05 6.72124316e-05 6.73435909e-05 6.74602098e-05
 8.00358561e-05 8.11030008e-05 8.11030008e-05 8.29462508e-05
 8.44822925e-05 8.68960723e-05 8.87064071e-05 8.87064071e-05
 8.87064071e-05 8.87064071e-05 8.87064071e-05 8.94317602e-05
 8.96812028e-05 9.07982102e-05 9.21625009e-05 9.21625009e-05
 9.29305218e-05 9.29305218e-05 9.35803856e-05 9.35803856e-05
 9.35803856e-05 9.38692139e-05 9.50425791e-05 9.50425791e-05
 9.54152951e-05 9.54152951e-05 9.54152951e-05 9.57465982e-05
 9.57465982e-05 9.60178457e-05 9.60430273e-05 9.60430273e-05
 9.63098135e-05 9.63098135e-05 9.65511914e-05 9.67706260e-05
 9.72408428e-05 9.74795683e-05 9.76239825e-05 9.78829320e-05
 9.79994593e-05 9.79994593e-05 9.80012253e-05 9.81084687e-05
 9.83066676e-05 9.83525196e-05 9.85850319e-05 9.87088721e-05
 9.87108891e-05 9.88442822e-05 9.89060986e-05 9.90211056e-05
 9.94291596e-05 1.000087

In [14]:
# Grid Search CV :

from sklearn.model_selection import GridSearchCV

model = DecisionTreeClassifier(random_state=42)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ]
)

param_grid = {
    "model__ccp_alpha" : [0.0,0.001,0.005,0.01],
    "model__max_depth" : [2,3,4,5,6,7,8,9,10],
    "model__min_samples_leaf": [5,10,20,30,40,50]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)

In [15]:
grid.fit(x_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__ccp_alpha': [0.0, 0.001, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [5, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose:

In [18]:
print("Best Parameters : ",grid.best_params_)
print("Best CV F1 : ",grid.best_score_)

y_pred = grid.predict(x_test)
print("Test F! : ",f1_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

Best Parameters :  {'model__ccp_alpha': 0.0, 'model__max_depth': 5, 'model__min_samples_leaf': 30}
Best CV F1 :  0.6446911110260841
Test F! :  0.592375366568915
              precision    recall  f1-score   support

           0       0.90      0.97      0.93      2055
           1       0.75      0.49      0.59       411

    accuracy                           0.89      2466
   macro avg       0.83      0.73      0.76      2466
weighted avg       0.88      0.89      0.88      2466

[[1986   69]
 [ 209  202]]
